#Check Spark

In [ ]:
# Cài PySpark vào môi trường Colab hiện tại
%pip install -q pyspark

# Kiểm tra cài đặt
import pyspark

print("Cài PySpark thành công.")
print("Phiên bản:", pyspark.__version__)

Cài PySpark thành công.
Phiên bản: 4.0.5


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Khởi động Spark trên máy Colab hiện tại
spark = (
    SparkSession.builder
    .appName("HMDA_Thuy_RQ2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# Giảm các thông báo kỹ thuật không cần thiết
spark.sparkContext.setLogLevel("ERROR")

print("Khởi động Spark thành công.")
print("Phiên bản Spark:", spark.version)
print("Chế độ chạy:", spark.sparkContext.master)

# Tạo dữ liệu nhỏ để kiểm tra Spark hoạt động
test_df = spark.createDataFrame(
    [
        ("Originated", 100000),
        ("Originated", 200000),
        ("Denied", 90000),
    ],
    ["outcome", "loan_amount"]
)

print("\nDỮ LIỆU CHẠY THỬ")
test_df.show()

print("KẾT QUẢ TỔNG HỢP")
(
    test_df.groupBy("outcome")
    .agg(
        F.count("*").alias("applications"),
        F.avg("loan_amount").alias("average_loan_amount")
    )
    .orderBy("outcome")
    .show()
)

Khởi động Spark thành công.
Phiên bản Spark: 4.0.5
Chế độ chạy: local[*]

DỮ LIỆU CHẠY THỬ
+----------+-----------+
|   outcome|loan_amount|
+----------+-----------+
|Originated|     100000|
|Originated|     200000|
|    Denied|      90000|
+----------+-----------+

KẾT QUẢ TỔNG HỢP
+----------+------------+-------------------+
|   outcome|applications|average_loan_amount|
+----------+------------+-------------------+
|    Denied|           1|            90000.0|
|Originated|           2|           150000.0|
+----------+------------+-------------------+



In [ ]:
from google.colab import drive
from pathlib import Path

# Kết nối lại Drive nếu cần
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/hmda")

if not BASE.exists():
    print("CHƯA TÌM THẤY folder hmda.")
    print("Kiểm tra shortcut hmda trong My Drive và tài khoản vừa kết nối.")
else:
    print("CẤU TRÚC FOLDER HMDA")
    print("=" * 50)

    # Chỉ xem tên file/folder, không đọc toàn bộ dữ liệu
    for item in sorted(BASE.iterdir()):
        if item.name.startswith(".") or item.name == "__pycache__":
            continue

        kind = "FOLDER" if item.is_dir() else "FILE"
        print(f"\n[{kind}] {item.name}")

        # Hiển thị các folder con trực tiếp
        # Bỏ qua raw.parquet vì đây là folder chứa các mảnh dữ liệu
        if item.is_dir() and item.name != "raw.parquet":
            subfolders = sorted(
                child.name
                for child in item.iterdir()
                if child.is_dir()
                and not child.name.startswith(".")
                and child.name != "__pycache__"
            )
            for name in subfolders[:15]:
                print(f"    → {name}")

            if len(subfolders) > 15:
                print(f"    ... và {len(subfolders) - 15} folder con khác")

    print("\nKiểm tra xong. Chưa thay đổi dữ liệu nào.")

Mounted at /content/drive
CẤU TRÚC FOLDER HMDA

[FOLDER] audit

[FOLDER] audit_pandas

[FOLDER] clean_pandas
    → S0_all
    → S1_orig_denied
    → S2_denied

[FOLDER] raw.parquet

[FOLDER] rq1_pandas

[FILE] rules.py

Kiểm tra xong. Chưa thay đổi dữ liệu nào.


In [ ]:
from pyspark.sql import functions as F

# Hiện dùng bản Pandas đã cleaning để chuẩn bị phần phân tích Spark
S1_SPARK_PATH = "/content/drive/MyDrive/hmda/clean_pandas/S1_orig_denied"
SPARK_INPUT_SOURCE = "Pandas-cleaned data — analysis check only"

COLS = [
    "action_taken",
    "denied_flag",
    "income",
    "debt_to_income_ratio",
    "loan_amount",
    "combined_loan_to_value_ratio",
    "property_value",
    "dti_band",
    "cltv_band",
    "loan_to_income",
]

print("Đang đọc cấu trúc dữ liệu...")

source_df = spark.read.parquet(S1_SPARK_PATH)

missing_cols = [c for c in COLS if c not in source_df.columns]
if missing_cols:
    raise ValueError(f"Thiếu các cột: {missing_cols}")

# Bảng Spark dùng cho RQ2
s1_sp = source_df.select(*COLS)

# Chuẩn hóa missing số:
# Pandas có thể lưu missing dưới dạng NaN,
# trong khi Spark phân biệt NaN và null.
NUMERIC_COLS = [
    "income",
    "loan_amount",
    "combined_loan_to_value_ratio",
    "property_value",
    "loan_to_income",
]

for column in NUMERIC_COLS:
    value = F.col(column).cast("double")
    s1_sp = s1_sp.withColumn(
        column,
        F.when(value.isNull() | F.isnan(value), F.lit(None))
         .otherwise(value)
    )

s1_sp = (
    s1_sp
    .withColumn("action_taken", F.col("action_taken").cast("string"))
    .withColumn("denied_flag", F.col("denied_flag").cast("int"))
)

print("\nSCHEMA — TÊN CỘT VÀ KIỂU DỮ LIỆU")
s1_sp.printSchema()

print("\nĐang đếm và kiểm tra toàn bộ S1...")

valid_target = (
    ((F.col("action_taken") == "1") & (F.col("denied_flag") == 0))
    | ((F.col("action_taken") == "3") & (F.col("denied_flag") == 1))
)

check = s1_sp.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("action_taken") == "1", 1).otherwise(0))
        .alias("originated"),
    F.sum(F.when(F.col("action_taken") == "3", 1).otherwise(0))
        .alias("denied"),
    F.sum(
        F.when(F.coalesce(valid_target, F.lit(False)), 0).otherwise(1)
    ).alias("invalid_target_rows"),
).first()

rate = check["denied"] / check["total"] * 100

print("\nKẾT QUẢ KIỂM TRA")
print(f"Tổng số dòng:          {check['total']:,}")
print(f"Originated:            {check['originated']:,}")
print(f"Denied:                {check['denied']:,}")
print(f"Denial rate:           {rate:.2f}%")
print(f"Dòng có target sai:    {check['invalid_target_rows']:,}")
print(f"Nguồn dữ liệu:         {SPARK_INPUT_SOURCE}")

assert check["total"] == 8_300_547, "Số dòng chưa khớp!"
assert check["originated"] == 6_197_085, "Số originated chưa khớp!"
assert check["denied"] == 2_103_462, "Số denied chưa khớp!"
assert check["invalid_target_rows"] == 0, "Có target không hợp lệ!"

print("\nOK — Spark đọc đúng S1, sẵn sàng chạy phân tích RQ2.")

Đang đọc cấu trúc dữ liệu...

SCHEMA — TÊN CỘT VÀ KIỂU DỮ LIỆU
root
 |-- action_taken: string (nullable = true)
 |-- denied_flag: integer (nullable = true)
 |-- income: double (nullable = true)
 |-- debt_to_income_ratio: string (nullable = true)
 |-- loan_amount: double (nullable = true)
 |-- combined_loan_to_value_ratio: double (nullable = true)
 |-- property_value: double (nullable = true)
 |-- dti_band: string (nullable = true)
 |-- cltv_band: string (nullable = true)
 |-- loan_to_income: double (nullable = true)


Đang đếm và kiểm tra toàn bộ S1...

KẾT QUẢ KIỂM TRA
Tổng số dòng:          8,300,547
Originated:            6,197,085
Denied:                2,103,462
Denial rate:           25.34%
Dòng có target sai:    0
Nguồn dữ liệu:         Pandas-cleaned data — analysis check only

OK — Spark đọc đúng S1, sẵn sàng chạy phân tích RQ2.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/hmda")

paths = {
    "S1 được cleaning bằng Spark":
        BASE / "clean_spark" / "S1_orig_denied",

    "Schema Spark do Vanh xuất":
        BASE / "audit_spark" / "schema_clean_spark.csv",

    "Bảng Pandas của Thủy":
        Path("/content/drive/MyDrive/HMDA_Thuy/rq2_pandas/tables"),
}

print("\nKIỂM TRA DỮ LIỆU BÀN GIAO")
print("-" * 55)

for name, path in paths.items():
    print(f"{'OK' if path.exists() else 'CHƯA TÌM THẤY'} | {name}")

print("\nChưa đọc toàn bộ dữ liệu và chưa thay đổi file nào.")

Mounted at /content/drive

KIỂM TRA DỮ LIỆU BÀN GIAO
-------------------------------------------------------
OK | S1 được cleaning bằng Spark
OK | Schema Spark do Vanh xuất
OK | Bảng Pandas của Thủy

Chưa đọc toàn bộ dữ liệu và chưa thay đổi file nào.


In [ ]:
%pip install -q pyspark==4.0.5

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
import pandas as pd
from IPython.display import display

# 1. Khởi động Spark
spark = (
    SparkSession.builder
    .appName("HMDA_Thuy_RQ2_Official")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

# 2. Đọc đúng bản cleaning bằng Spark
S1_SPARK_PATH = (
    "/content/drive/MyDrive/hmda/clean_spark/S1_orig_denied"
)
SPARK_INPUT_SOURCE = "clean_spark/S1_orig_denied"

COLS = [
    "action_taken", "denied_flag",
    "income", "debt_to_income_ratio",
    "loan_amount", "combined_loan_to_value_ratio",
    "property_value", "dti_band", "cltv_band",
    "loan_to_income",
]

if not Path(S1_SPARK_PATH).exists():
    raise FileNotFoundError("Chưa tìm thấy S1 Spark. Kiểm tra kết nối Drive.")

source_df = spark.read.parquet(S1_SPARK_PATH)

missing_cols = [c for c in COLS if c not in source_df.columns]
if missing_cols:
    raise ValueError(f"Thiếu các cột: {missing_cols}")

s1_sp = source_df.select(*COLS)

# Chuẩn hóa missing số để các phép tính bỏ qua cả NaN và null
numeric_cols = [
    "income", "loan_amount",
    "combined_loan_to_value_ratio",
    "property_value", "loan_to_income",
]

for column in numeric_cols:
    value = F.col(column).cast("double")
    s1_sp = s1_sp.withColumn(
        column,
        F.when(value.isNull() | F.isnan(value), F.lit(None))
         .otherwise(value)
    )

# 3. Kiểm tra số hồ sơ và target trong một lần tổng hợp
valid_target = (
    ((F.col("action_taken") == "1") & (F.col("denied_flag") == 0))
    | ((F.col("action_taken") == "3") & (F.col("denied_flag") == 1))
)

quality_cols = [
    "income", "debt_to_income_ratio", "loan_amount",
    "combined_loan_to_value_ratio", "property_value",
    "dti_band", "cltv_band",
]

expressions = [
    F.count("*").alias("total"),
    F.sum(F.when(F.col("action_taken") == "1", 1).otherwise(0))
        .alias("originated"),
    F.sum(F.when(F.col("action_taken") == "3", 1).otherwise(0))
        .alias("denied"),
    F.sum(
        F.when(F.coalesce(valid_target, F.lit(False)), 0).otherwise(1)
    ).alias("invalid_target_rows"),
]

expressions += [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0))
     .alias(f"missing_{c}")
    for c in quality_cols
]

print("\nĐang kiểm tra toàn bộ S1 Spark...")
check = s1_sp.agg(*expressions).first().asDict()

print("\nKẾT QUẢ KIỂM TRA")
print(f"Nguồn:             {SPARK_INPUT_SOURCE}")
print(f"Tổng hồ sơ:        {check['total']:,}")
print(f"Originated:        {check['originated']:,}")
print(f"Denied:            {check['denied']:,}")
print(f"Denial rate:       {check['denied'] / check['total'] * 100:.2f}%")
print(f"Dòng target sai:   {check['invalid_target_rows']:,}")

# 4. Đối chiếu missing với bản Pandas đã kiểm tra trước đó
expected_missing = {
    "income": 624510,
    "debt_to_income_ratio": 883734,
    "loan_amount": 0,
    "combined_loan_to_value_ratio": 843490,
    "property_value": 534689,
    "dti_band": 883734,
    "cltv_band": 843490,
}

rq2_missing_check = pd.DataFrame([
    {
        "variable": c,
        "pandas_missing": expected_missing[c],
        "spark_missing": check[f"missing_{c}"],
        "match": expected_missing[c] == check[f"missing_{c}"],
    }
    for c in quality_cols
])

print("\nĐỐI CHIẾU SỐ GIÁ TRỊ THIẾU")
display(rq2_missing_check)

counts_ok = (
    check["total"] == 8_300_547
    and check["originated"] == 6_197_085
    and check["denied"] == 2_103_462
    and check["invalid_target_rows"] == 0
)

if counts_ok and rq2_missing_check["match"].all():
    print("\nPASS — số hồ sơ, target và missing khớp bản Pandas.")
else:
    print("\nCÓ CHÊNH LỆCH — gửi output để kiểm tra trước khi làm tiếp.")

Spark version: 4.0.5

Đang kiểm tra toàn bộ S1 Spark...

KẾT QUẢ KIỂM TRA
Nguồn:             clean_spark/S1_orig_denied
Tổng hồ sơ:        8,300,547
Originated:        6,197,085
Denied:            2,103,462
Denial rate:       25.34%
Dòng target sai:   0

ĐỐI CHIẾU SỐ GIÁ TRỊ THIẾU


,variable,pandas_missing,spark_missing,match
0,income,624510,624510,True
1,debt_to_income_ratio,883734,883734,True
2,loan_amount,0,0,True
3,combined_loan_to_value_ratio,843490,843490,True
4,property_value,534689,534689,True
5,dti_band,883734,883734,True
6,cltv_band,843490,843490,True



PASS — số hồ sơ, target và missing khớp bản Pandas.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from pyspark.sql import functions as F
from IPython.display import display

PANDAS_TABLES = Path(
    "/content/drive/MyDrive/HMDA_Thuy/rq2_pandas/tables"
)

DTI_ORDER = [
    "<20%", "20-<30%", "30-<36%", "36-43%",
    "44-49%", "50-60%", ">60%", "Missing"
]
CLTV_ORDER = [
    "<=60", "60-80", "80-90",
    "90-95", "95-100", "100-250", "Missing"
]

def calculate_and_compare(column, order, pandas_filename):
    # 1. Spark tổng hợp trên bản clean_spark
    result = (
        s1_sp
        .groupBy(
            F.coalesce(F.col(column), F.lit("Missing")).alias(column)
        )
        .agg(
            F.count("*").alias("applications"),
            F.sum("denied_flag").alias("denied")
        )
        .withColumn(
            "originated", F.col("applications") - F.col("denied")
        )
        .withColumn(
            "denial_rate_pct",
            F.col("denied") / F.col("applications") * 100
        )
        .withColumn(
            "share_of_S1_pct",
            F.col("applications") / F.lit(check["total"]) * 100
        )
        .toPandas()  # Chỉ lấy bảng tổng hợp nhỏ về Python
        .set_index(column)
    )

    # 2. Đọc bảng Pandas đã lưu
    reference = pd.read_csv(
        PANDAS_TABLES / pandas_filename, index_col=0
    )

    # Kiểm tra nhóm trước khi sắp xếp
    if not result.index.is_unique or not reference.index.is_unique:
        raise ValueError("Có nhãn nhóm bị trùng.")

    if set(result.index) != set(reference.index):
        raise ValueError(
            f"{column}: danh sách nhóm giữa hai công cụ không khớp."
        )

    if set(result.index) - set(order):
        raise ValueError(f"{column}: có nhóm ngoài danh sách dự kiến.")

    labels = [label for label in order if label in result.index]
    result = result.reindex(labels)
    reference = reference.reindex(labels)

    # 3. Đối chiếu từng chỉ số trên tất cả các nhóm
    comparisons = []

    for metric in [
        "applications", "denied", "originated",
        "denial_rate_pct", "share_of_S1_pct"
    ]:
        p = reference[metric]
        s = result[metric]

        if metric in ["applications", "denied", "originated"]:
            matched = p.eq(s).all()
        else:
            matched = np.isclose(
                p.to_numpy(dtype=float),
                s.to_numpy(dtype=float),
                atol=1e-9,
                rtol=0,
                equal_nan=True
            ).all()

        comparisons.append({
            "analysis": column,
            "metric": metric,
            "groups_checked": len(labels),
            "max_abs_difference": (p - s).abs().max(),
            "match": bool(matched),
        })

    return result, pd.DataFrame(comparisons)


print("Đang tính DTI bằng Spark...")
rq2_dti_sp, dti_checks = calculate_and_compare(
    "dti_band", DTI_ORDER, "04_denial_by_dti.csv"
)

print("Đang tính CLTV bằng Spark...")
rq2_cltv_sp, cltv_checks = calculate_and_compare(
    "cltv_band", CLTV_ORDER, "05_denial_by_cltv.csv"
)

print("\nBẢNG SPARK — DTI")
display(rq2_dti_sp.round(2))

print("\nBẢNG SPARK — CLTV")
display(rq2_cltv_sp.round(2))

rq2_band_checks = pd.concat(
    [dti_checks, cltv_checks], ignore_index=True
)

print("\nĐỐI CHIẾU PANDAS — SPARK CHÍNH THỨC")
display(rq2_band_checks)

if rq2_band_checks["match"].all():
    print("\nPASS — các nhóm DTI và CLTV đều khớp Pandas.")
else:
    print("\nCÓ CHÊNH LỆCH — gửi bảng kiểm tra trước khi làm tiếp.")

print("\nNguồn Spark:", S1_SPARK_PATH)

Đang tính DTI bằng Spark...
Đang tính CLTV bằng Spark...

BẢNG SPARK — DTI


,applications,denied,originated,denial_rate_pct,share_of_S1_pct
dti_band,,,,,
<20%,432229,131641,300588,30.46,5.21
20-<30%,999770,166429,833341,16.65,12.04
30-<36%,1026750,152255,874495,14.83,12.37
36-43%,1951343,279011,1672332,14.30,23.51
44-49%,1625066,254625,1370441,15.67,19.58
50-60%,777872,311435,466437,40.04,9.37
>60%,603783,556282,47501,92.13,7.27
Missing,883734,251784,631950,28.49,10.65



BẢNG SPARK — CLTV


,applications,denied,originated,denial_rate_pct,share_of_S1_pct
cltv_band,,,,,
<=60,1879464,545472,1333992,29.02,22.64
60-80,2783553,604959,2178594,21.73,33.53
80-90,861463,218882,642581,25.41,10.38
90-95,556162,101000,455162,18.16,6.70
95-100,1081004,167838,913166,15.53,13.02
100-250,295411,89184,206227,30.19,3.56
Missing,843490,376127,467363,44.59,10.16



ĐỐI CHIẾU PANDAS — SPARK CHÍNH THỨC


,analysis,metric,groups_checked,max_abs_difference,match
0,dti_band,applications,8,0.000000e+00,True
1,dti_band,denied,8,0.000000e+00,True
2,dti_band,originated,8,0.000000e+00,True
3,dti_band,denial_rate_pct,8,1.776357e-15,True
4,dti_band,share_of_S1_pct,8,3.552714e-15,True
5,cltv_band,applications,7,0.000000e+00,True
6,cltv_band,denied,7,0.000000e+00,True
7,cltv_band,originated,7,0.000000e+00,True
8,cltv_band,denial_rate_pct,7,0.000000e+00,True
9,cltv_band,share_of_S1_pct,7,1.776357e-15,True



PASS — các nhóm DTI và CLTV đều khớp Pandas.

Nguồn Spark: /content/drive/MyDrive/hmda/clean_spark/S1_orig_denied


# Spark

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from IPython.display import display

VARIABLES = {
    "income": ("Income", "Thousand USD/year"),
    "loan_amount": ("Loan amount", "USD"),
    "combined_loan_to_value_ratio": ("CLTV", "%"),
    "property_value": ("Property value", "USD"),
}

parts = []

# Tính lần lượt từng biến để hạn chế bộ nhớ sử dụng cùng lúc
for column, (label, unit) in VARIABLES.items():
    print(f"Đang tính thống kê chính xác: {label}...")

    small = (
        s1_sp.groupBy("denied_flag")
        .agg(
            F.count("*").alias("total_records"),
            F.count(column).alias("valid_records"),
            F.expr(
                f"percentile({column}, array(0.25, 0.50, 0.75))"
            ).alias("quartiles")
        )
        .collect()  # Chỉ lấy 2 dòng kết quả tổng hợp
    )

    rows = []
    for row in small:
        q1, median, q3 = row["quartiles"]
        total = row["total_records"]
        valid = row["valid_records"]

        rows.append({
            "variable": label,
            "unit": unit,
            "outcome": (
                "Denied" if row["denied_flag"] == 1 else "Originated"
            ),
            "total_records": total,
            "valid_records": valid,
            "missing_pct": (total - valid) / total * 100,
            "Q1": q1,
            "median": median,
            "Q3": q3,
            "IQR": q3 - q1,
        })

    parts.append(pd.DataFrame(rows))

rq2_summary_sp = (
    pd.concat(parts, ignore_index=True)
    .sort_values(["variable", "outcome"])
    .reset_index(drop=True)
)

print("\nBẢNG THỐNG KÊ TÀI CHÍNH — SPARK")
display(rq2_summary_sp.round(2))

# Đọc bản Pandas đã lưu
reference = pd.read_csv(
    PANDAS_TABLES / "01_financial_summary.csv"
)

keys = ["variable", "unit", "outcome"]

# Ghép theo tên biến + đơn vị + outcome, không dựa vào thứ tự dòng
merged = reference.merge(
    rq2_summary_sp,
    on=keys,
    how="outer",
    suffixes=("_pandas", "_spark"),
    indicator=True,
    validate="one_to_one",
)

if not merged["_merge"].eq("both").all():
    raise ValueError("Danh sách biến/outcome của hai bảng không khớp.")

comparisons = []

for metric in [
    "total_records", "valid_records",
    "missing_pct", "Q1", "median", "Q3", "IQR"
]:
    p = merged[f"{metric}_pandas"]
    s = merged[f"{metric}_spark"]

    if metric in ["total_records", "valid_records"]:
        matches = p.eq(s).to_numpy()
    else:
        # Cho phép sai số số thực rất nhỏ; không làm tròn trước khi so
        matches = np.isclose(
            p.to_numpy(dtype=float),
            s.to_numpy(dtype=float),
            atol=1e-6,
            rtol=0,
            equal_nan=True,
        )

    comparisons.append({
        "metric": metric,
        "rows_checked": len(merged),
        "max_abs_difference": (p - s).abs().max(),
        "match": bool(matches.all()),
    })

    if not matches.all():
        print(f"\nCác dòng chưa khớp ở chỉ số {metric}:")
        display(
            merged.loc[
                ~matches,
                keys + [f"{metric}_pandas", f"{metric}_spark"]
            ]
        )

rq2_summary_checks = pd.DataFrame(comparisons)

print("\nĐỐI CHIẾU PANDAS — SPARK")
display(rq2_summary_checks)

if rq2_summary_checks["match"].all():
    print("\nPASS — thống kê tài chính khớp Pandas.")
else:
    print("\nCÓ CHÊNH LỆCH — cần kiểm tra trước khi làm tiếp.")

Đang tính thống kê chính xác: Income...
Đang tính thống kê chính xác: Loan amount...
Đang tính thống kê chính xác: CLTV...
Đang tính thống kê chính xác: Property value...

BẢNG THỐNG KÊ TÀI CHÍNH — SPARK


,variable,unit,outcome,total_records,valid_records,missing_pct,Q1,median,Q3,IQR
0,CLTV,%,Denied,2103462,1727335,17.88,53.86,74.51,87.38,33.52
1,CLTV,%,Originated,6197085,5729722,7.54,62.29,79.98,95.00,32.71
2,Income,Thousand USD/year,Denied,2103462,1950330,7.28,51.00,82.00,131.00,80.00
3,Income,Thousand USD/year,Originated,6197085,5725707,7.61,76.00,114.00,176.00,100.00
4,Loan amount,USD,Denied,2103462,2103462,0.00,65000.00,125000.00,255000.00,190000.00
5,Loan amount,USD,Originated,6197085,6197085,0.00,105000.00,235000.00,385000.00,280000.00
6,Property value,USD,Denied,2103462,1873348,10.94,205000.00,355000.00,555000.00,350000.00
7,Property value,USD,Originated,6197085,5892510,4.91,275000.00,395000.00,605000.00,330000.00



ĐỐI CHIẾU PANDAS — SPARK


,metric,rows_checked,max_abs_difference,match
0,total_records,8,0.000000e+00,True
1,valid_records,8,0.000000e+00,True
2,missing_pct,8,1.776357e-15,True
3,Q1,8,0.000000e+00,True
4,median,8,0.000000e+00,True
5,Q3,8,0.000000e+00,True
6,IQR,8,7.105427e-15,True



PASS — thống kê tài chính khớp Pandas.


In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from IPython.display import display

# 1. Spark tính lại từng tổ hợp DTI × CLTV
print("Đang tổng hợp DTI × CLTV bằng Spark...")

rq2_joint_sp = (
    s1_sp
    .groupBy(
        F.coalesce(
            F.col("dti_band"), F.lit("Missing")
        ).alias("dti_band"),
        F.coalesce(
            F.col("cltv_band"), F.lit("Missing")
        ).alias("cltv_band"),
    )
    .agg(
        F.count("*").alias("applications"),
        F.sum("denied_flag").alias("denied"),
    )
    .withColumn(
        "denial_rate_pct",
        F.col("denied") / F.col("applications") * 100
    )
    .toPandas()  # Chỉ lấy bảng tổng hợp nhỏ
)

# 2. Đọc kết quả Pandas đã lưu
reference = pd.read_csv(
    PANDAS_TABLES / "06_dti_cltv_joint_results.csv"
)

keys = ["dti_band", "cltv_band"]

merged_joint = reference.merge(
    rq2_joint_sp,
    on=keys,
    how="outer",
    suffixes=("_pandas", "_spark"),
    indicator=True,
    validate="one_to_one",
)

# Không để sót trường hợp một engine thiếu một tổ hợp nhóm
if not merged_joint["_merge"].eq("both").all():
    display(
        merged_joint.loc[merged_joint["_merge"] != "both"]
    )
    raise ValueError("Danh sách tổ hợp DTI × CLTV không khớp.")

# 3. Đối chiếu số đếm và tỷ lệ
comparisons = []

for metric in ["applications", "denied", "denial_rate_pct"]:
    p = merged_joint[f"{metric}_pandas"]
    s = merged_joint[f"{metric}_spark"]

    if metric in ["applications", "denied"]:
        matches = p.eq(s).to_numpy()
    else:
        matches = np.isclose(
            p.to_numpy(dtype=float),
            s.to_numpy(dtype=float),
            atol=1e-9,
            rtol=0,
            equal_nan=True,
        )

    comparisons.append({
        "metric": metric,
        "cells_checked": len(merged_joint),
        "max_abs_difference": (p - s).abs().max(),
        "match": bool(matches.all()),
    })

    if not matches.all():
        print(f"\nCác ô chưa khớp ở {metric}:")
        display(
            merged_joint.loc[
                ~matches,
                keys + [f"{metric}_pandas", f"{metric}_spark"]
            ]
        )

rq2_joint_checks = pd.DataFrame(comparisons)

# 4. Kiểm tra tổng số hồ sơ để đảm bảo không bỏ sót
assert rq2_joint_sp["applications"].sum() == check["total"]
assert rq2_joint_sp["denied"].sum() == check["denied"]

print("\nĐỐI CHIẾU HEATMAP PANDAS — SPARK")
display(rq2_joint_checks)

print("\nÔ ĐÁNG CHÚ Ý: DTI 50–60%, CLTV 95–100%")
display(
    rq2_joint_sp.loc[
        rq2_joint_sp["dti_band"].eq("50-60%")
        & rq2_joint_sp["cltv_band"].eq("95-100")
    ].round(2)
)

if rq2_joint_checks["match"].all():
    print("\nPASS — toàn bộ ô của heatmap khớp Pandas.")
else:
    print("\nCÓ CHÊNH LỆCH — cần kiểm tra trước khi làm tiếp.")

Đang tổng hợp DTI × CLTV bằng Spark...

ĐỐI CHIẾU HEATMAP PANDAS — SPARK


,metric,cells_checked,max_abs_difference,match
0,applications,56,0.000000e+00,True
1,denied,56,0.000000e+00,True
2,denial_rate_pct,56,1.421085e-14,True



Ô ĐÁNG CHÚ Ý: DTI 50–60%, CLTV 95–100%


,dti_band,cltv_band,applications,denied,denial_rate_pct
54,50-60%,95-100,238102,36094,15.16



PASS — toàn bộ ô của heatmap khớp Pandas.


In [ ]:
from pathlib import Path
import pandas as pd
import json

OUT_SPARK = Path(
    "/content/drive/MyDrive/HMDA_Thuy/rq2_spark"
)
TABLES_SPARK = OUT_SPARK / "tables"
CHECKS_SPARK = OUT_SPARK / "correctness"

TABLES_SPARK.mkdir(parents=True, exist_ok=True)
CHECKS_SPARK.mkdir(parents=True, exist_ok=True)

# 1. Các bảng phân tích được tính bằng Spark
tables = {
    "01_financial_summary.csv": (rq2_summary_sp, False),
    "04_denial_by_dti.csv": (rq2_dti_sp, True),
    "05_denial_by_cltv.csv": (rq2_cltv_sp, True),
    "06_dti_cltv_joint_results.csv": (rq2_joint_sp, False),
}

for filename, (table, keep_index) in tables.items():
    path = TABLES_SPARK / filename
    table.to_csv(path, index=keep_index, encoding="utf-8-sig")

    assert len(pd.read_csv(path)) == len(table)
    print(f"Đã lưu bảng: {filename}")

# 2. Bằng chứng đối chiếu Pandas–Spark
checks_to_save = {
    "01_missing_check.csv": rq2_missing_check,
    "02_band_checks.csv": rq2_band_checks,
    "03_financial_summary_checks.csv": rq2_summary_checks,
    "04_joint_heatmap_checks.csv": rq2_joint_checks,
}

for filename, table in checks_to_save.items():
    path = CHECKS_SPARK / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")

    assert len(pd.read_csv(path)) == len(table)
    print(f"Đã lưu kiểm tra: {filename}")

# 3. Ghi rõ nguồn dữ liệu và phạm vi đã kiểm tra
metadata = {
    "spark_version": spark.version,
    "spark_input": S1_SPARK_PATH,
    "pandas_reference": str(PANDAS_TABLES),
    "total_records": int(check["total"]),
    "originated": int(check["originated"]),
    "denied": int(check["denied"]),
    "invalid_target_rows": int(check["invalid_target_rows"]),
    "all_saved_comparisons_pass": bool(
        rq2_missing_check["match"].all()
        and rq2_band_checks["match"].all()
        and rq2_summary_checks["match"].all()
        and rq2_joint_checks["match"].all()
    ),
    "quantile_method": "Exact percentile",
    "comparison_tolerances": {
        "counts": "Exact equality",
        "band_and_joint_rates_percentage_points": 1e-9,
        "financial_summary_numeric_values": 1e-6,
    },
    "pending": ["Pairwise Spearman comparison"],
}

with open(OUT_SPARK / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\nHOÀN THÀNH — đã lưu 8 file CSV và 1 file run_info.json.")
print("Vị trí: My Drive → HMDA_Thuy → rq2_spark")
print("Các đối chiếu đã lưu đều PASS:",
      metadata["all_saved_comparisons_pass"])
print("Còn lại: đối chiếu Spearman.")

Đã lưu bảng: 01_financial_summary.csv
Đã lưu bảng: 04_denial_by_dti.csv
Đã lưu bảng: 05_denial_by_cltv.csv
Đã lưu bảng: 06_dti_cltv_joint_results.csv
Đã lưu kiểm tra: 01_missing_check.csv
Đã lưu kiểm tra: 02_band_checks.csv
Đã lưu kiểm tra: 03_financial_summary_checks.csv
Đã lưu kiểm tra: 04_joint_heatmap_checks.csv

HOÀN THÀNH — đã lưu 8 file CSV và 1 file run_info.json.
Vị trí: My Drive → HMDA_Thuy → rq2_spark
Các đối chiếu đã lưu đều PASS: True
Còn lại: đối chiếu Spearman.


In [ ]:
import math
import numpy as np
import pandas as pd
from itertools import combinations
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel
from IPython.display import display

CORR_COLS = [
    "income",
    "loan_amount",
    "combined_loan_to_value_ratio",
    "property_value",
    "denied_flag",
]

# Đọc kết quả Pandas chưa làm tròn
reference_corr = pd.read_csv(
    PANDAS_TABLES / "09_spearman_matrix.csv", index_col=0
)
reference_n = pd.read_csv(
    PANDAS_TABLES / "10_spearman_pair_counts.csv", index_col=0
)

def spearman_pair_spark(col_a, col_b):
    # Chỉ giữ hồ sơ có đủ dữ liệu cho cặp đang xét
    pair = (
        s1_sp.select(
            F.col(col_a).cast("double").alias("x"),
            F.col(col_b).cast("double").alias("y"),
        )
        .filter(
            F.col("x").isNotNull()
            & F.col("y").isNotNull()
            & ~F.isnan("x")
            & ~F.isnan("y")
        )
    )

    # Gom các cặp giá trị giống nhau và giữ nguyên tần suất.
    # Không lấy mẫu; mỗi hồ sơ vẫn được tính thông qua trọng số w.
    freq = (
        pair.groupBy("x", "y")
        .agg(F.count("*").alias("w"))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    try:
        n = int(freq.agg(F.sum("w")).first()[0])

        if n < 2:
            return float("nan"), n

        def average_ranks(column, rank_name):
            marginal = (
                freq.groupBy(column)
                .agg(F.sum("w").alias("frequency"))
            )

            ordered = (
                Window.orderBy(column)
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )

            # Các giá trị bằng nhau nhận thứ hạng trung bình
            return marginal.select(
                column,
                (
                    F.sum("frequency").over(ordered)
                    - (F.col("frequency") - 1) / 2.0
                ).alias(rank_name),
            )

        x_ranks = average_ranks("x", "rx")
        y_ranks = average_ranks("y", "ry")

        ranked = (
            freq.join(x_ranks, "x")
                .join(y_ranks, "y")
        )

        # Spearman = tương quan Pearson giữa các thứ hạng.
        # Trung bình thứ hạng luôn là (n + 1) / 2.
        center = (n + 1) / 2.0
        dx = F.col("rx") - F.lit(center)
        dy = F.col("ry") - F.lit(center)
        weight = F.col("w").cast("double")

        sums = ranked.agg(
            F.sum(weight * dx * dy).alias("cross"),
            F.sum(weight * dx * dx).alias("xx"),
            F.sum(weight * dy * dy).alias("yy"),
        ).first()

        denominator = math.sqrt(sums["xx"]) * math.sqrt(sums["yy"])
        rho = (
            sums["cross"] / denominator
            if denominator > 0 else float("nan")
        )

        return rho, n

    finally:
        freq.unpersist()


# Tính và kiểm tra lần lượt 10 cặp
pairs = list(combinations(CORR_COLS, 2))
results = []

for i, (a, b) in enumerate(pairs, start=1):
    print(f"[{i}/{len(pairs)}] Đang tính: {a} × {b}", flush=True)

    rho, n = spearman_pair_spark(a, b)
    pandas_rho = float(reference_corr.loc[a, b])
    pandas_n = int(reference_n.loc[a, b])

    results.append({
        "variable_1": a,
        "variable_2": b,
        "pandas_rho": pandas_rho,
        "spark_rho": rho,
        "abs_difference": abs(pandas_rho - rho),
        "pandas_n": pandas_n,
        "spark_n": n,
        "count_match": n == pandas_n,
        "correlation_match": bool(np.isclose(
            pandas_rho, rho,
            atol=1e-8, rtol=0, equal_nan=True
        )),
    })

    # Lưu tiến độ sau mỗi cặp để không mất kết quả đã tính
    pd.DataFrame(results).to_csv(
        CHECKS_SPARK / "05_spearman_pair_checks.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(f"    Xong: rho={rho:.6f}, n={n:,}", flush=True)

rq2_spearman_checks = pd.DataFrame(results)

print("\nĐỐI CHIẾU SPEARMAN PANDAS — SPARK")
display(rq2_spearman_checks)

passed = (
    rq2_spearman_checks["count_match"].all()
    and rq2_spearman_checks["correlation_match"].all()
)

if passed:
    print("\nPASS — cả 10 cặp khớp hệ số và số hồ sơ.")
else:
    print("\nCÓ CHÊNH LỆCH — gửi bảng đối chiếu để kiểm tra.")

print("\nĐã lưu: rq2_spark/correctness/05_spearman_pair_checks.csv")

[1/10] Đang tính: income × loan_amount
    Xong: rho=0.460452, n=7,676,037
[2/10] Đang tính: income × combined_loan_to_value_ratio
    Xong: rho=0.006363, n=7,018,034
[3/10] Đang tính: income × property_value
    Xong: rho=0.630999, n=7,300,715
[4/10] Đang tính: income × denied_flag
    Xong: rho=-0.212633, n=7,676,037
[5/10] Đang tính: loan_amount × combined_loan_to_value_ratio
    Xong: rho=0.307432, n=7,457,057
[6/10] Đang tính: loan_amount × property_value
    Xong: rho=0.519861, n=7,765,858
[7/10] Đang tính: loan_amount × denied_flag
    Xong: rho=-0.233541, n=8,300,547
[8/10] Đang tính: combined_loan_to_value_ratio × property_value
    Xong: rho=-0.246469, n=7,387,807
[9/10] Đang tính: combined_loan_to_value_ratio × denied_flag
    Xong: rho=-0.095278, n=7,457,057
[10/10] Đang tính: property_value × denied_flag
    Xong: rho=-0.096821, n=7,765,858

ĐỐI CHIẾU SPEARMAN PANDAS — SPARK


,variable_1,variable_2,pandas_rho,spark_rho,abs_difference,pandas_n,spark_n,count_match,correlation_match
0,income,loan_amount,0.460452,0.460452,3.247624e-12,7676037,7676037,True,True
1,income,combined_loan_to_value_ratio,0.006363,0.006363,1.503190e-13,7018034,7018034,True,True
2,income,property_value,0.630999,0.630999,9.601986e-12,7300715,7300715,True,True
3,income,denied_flag,-0.212633,-0.212633,5.496770e-12,7676037,7676037,True,True
4,loan_amount,combined_loan_to_value_ratio,0.307432,0.307432,2.296996e-12,7457057,7457057,True,True
5,loan_amount,property_value,0.519861,0.519861,2.599698e-12,7765858,7765858,True,True
6,loan_amount,denied_flag,-0.233541,-0.233541,1.691924e-11,8300547,8300547,True,True
7,combined_loan_to_value_ratio,property_value,-0.246469,-0.246469,1.225353e-12,7387807,7387807,True,True
8,combined_loan_to_value_ratio,denied_flag,-0.095278,-0.095278,2.647674e-12,7457057,7457057,True,True
9,property_value,denied_flag,-0.096821,-0.096821,2.284700e-12,7765858,7765858,True,True



PASS — cả 10 cặp khớp hệ số và số hồ sơ.

Đã lưu: rq2_spark/correctness/05_spearman_pair_checks.csv


In [ ]:
import pandas as pd
import numpy as np
import json
from IPython.display import display

# 1. Đảm bảo đủ 10 cặp và tất cả đều khớp
assert len(rq2_spearman_checks) == 10, "Chưa tính đủ 10 cặp."
assert rq2_spearman_checks["count_match"].all(), "Số hồ sơ chưa khớp."
assert rq2_spearman_checks["correlation_match"].all(), "Hệ số chưa khớp."

# 2. Tạo ma trận từ các hệ số thực tế Spark đã tính
rq2_spearman_sp = pd.DataFrame(
    np.nan, index=CORR_COLS, columns=CORR_COLS
)
rq2_corr_n_sp = pd.DataFrame(
    0, index=CORR_COLS, columns=CORR_COLS, dtype="int64"
)

for row in rq2_spearman_checks.itertuples(index=False):
    a, b = row.variable_1, row.variable_2
    rq2_spearman_sp.loc[a, b] = row.spark_rho
    rq2_spearman_sp.loc[b, a] = row.spark_rho
    rq2_corr_n_sp.loc[a, b] = row.spark_n
    rq2_corr_n_sp.loc[b, a] = row.spark_n

# Đếm số giá trị hợp lệ và kiểm tra biến không phải hằng số
diagonal_exprs = []
for c in CORR_COLS:
    diagonal_exprs.extend([
        F.count(c).alias(f"{c}_n"),
        F.min(c).alias(f"{c}_min"),
        F.max(c).alias(f"{c}_max"),
    ])

diagonal = s1_sp.agg(*diagonal_exprs).first().asDict()

for c in CORR_COLS:
    n = diagonal[f"{c}_n"]
    rq2_corr_n_sp.loc[c, c] = n

    if n >= 2 and diagonal[f"{c}_min"] != diagonal[f"{c}_max"]:
        rq2_spearman_sp.loc[c, c] = 1.0

# 3. Lưu bảng kết quả Spark
rq2_spearman_sp.to_csv(
    TABLES_SPARK / "09_spearman_matrix.csv",
    encoding="utf-8-sig"
)
rq2_corr_n_sp.to_csv(
    TABLES_SPARK / "10_spearman_pair_counts.csv",
    encoding="utf-8-sig"
)

rq2_spearman_checks.to_csv(
    CHECKS_SPARK / "05_spearman_pair_checks.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Tạo bảng tổng kết correctness của RQ2
rq2_correctness_summary = pd.DataFrame([
    {
        "output": "Sample counts and target",
        "pass": bool(
            check["total"] == 8_300_547
            and check["originated"] == 6_197_085
            and check["denied"] == 2_103_462
            and check["invalid_target_rows"] == 0
        ),
    },
    {
        "output": "Missing counts",
        "pass": bool(rq2_missing_check["match"].all()),
    },
    {
        "output": "Denial rates by DTI and CLTV",
        "pass": bool(rq2_band_checks["match"].all()),
    },
    {
        "output": "Financial summary and exact quartiles",
        "pass": bool(rq2_summary_checks["match"].all()),
    },
    {
        "output": "DTI × CLTV joint analysis",
        "pass": bool(rq2_joint_checks["match"].all()),
    },
    {
        "output": "Spearman: 10 pairs and pair counts",
        "pass": bool(
            rq2_spearman_checks["count_match"].all()
            and rq2_spearman_checks["correlation_match"].all()
        ),
    },
])

rq2_correctness_summary.to_csv(
    CHECKS_SPARK / "00_correctness_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. Cập nhật thông tin lần chạy
info_path = OUT_SPARK / "run_info.json"

with open(info_path, encoding="utf-8") as f:
    info = json.load(f)

info["pending"] = []
info["all_saved_comparisons_pass"] = bool(
    rq2_correctness_summary["pass"].all()
)
info["spearman_method"] = (
    "Pairwise complete observations; average ranks for ties; "
    "frequency-weighted rank correlation computed in Spark"
)
info["comparison_tolerances"]["spearman_absolute"] = 1e-8

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("TỔNG KẾT CORRECTNESS RQ2")
display(rq2_correctness_summary)

print("\nĐã lưu ma trận Spearman, số hồ sơ và bảng tổng kết.")
print("Vị trí: My Drive → HMDA_Thuy → rq2_spark")

TỔNG KẾT CORRECTNESS RQ2


,output,pass
0,Sample counts and target,True
1,Missing counts,True
2,Denial rates by DTI and CLTV,True
3,Financial summary and exact quartiles,True
4,DTI × CLTV joint analysis,True
5,Spearman: 10 pairs and pair counts,True



Đã lưu ma trận Spearman, số hồ sơ và bảng tổng kết.
Vị trí: My Drive → HMDA_Thuy → rq2_spark
